# Sistema de Detecção de Intrusão (IDS) com Deep Learning

Este notebook tem como objetivo construir um modelo de Deep Learning utilizando o dataset **CIC-IDS2018**. 
O processo consistirá em carregar os dados brutos, realizar a limpeza, pré-processamento e dividir o conjunto total em **Treino, Validação e Teste**.

---
## 1. Configuração do Ambiente e Importações


In [11]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import os
import gc # Para limpeza de memória RAM

# Desativar avisos desnecessários
import warnings
warnings.filterwarnings('ignore')

print("TensorFlow versão:", tf.__version__)
print("Ambiente configurado com sucesso!")

TensorFlow versão: 2.21.0
Ambiente configurado com sucesso!


## 2. Carregamento e Unificação dos Dados
Nesta etapa, leremos os 4 arquivos CSV presentes na pasta `archive` e os unificaremos em um único DataFrame. 
Como esses arquivos são grandes, usaremos o `gc.collect()` para liberar memória RAM após a unificação.

In [12]:
folder = 'archive'
# Lista exata dos arquivos na sua pasta
arquivos = ['02-14-2018.csv', '02-23-2018.csv', '03-01-2018.csv', '03-02-2018.csv']

df_list = []

for f in arquivos:
    path = os.path.join(folder, f)
    print(f"Lendo: {f}...")
    # low_memory=False evita avisos de tipos mistos
    temp = pd.read_csv(path, low_memory=False)
    
    # Limpeza crucial: remove linhas que repetem o cabeçalho no meio dos dados
    temp = temp[temp.iloc[:, 0] != temp.columns[0]]
    
    df_list.append(temp)

# Unificar tudo
df = pd.concat(df_list, ignore_index=True)

# Limpeza de memória
del df_list
gc.collect()

print(f"\nSucesso! Total de linhas carregadas: {df.shape[0]}")

Lendo: 02-14-2018.csv...
Lendo: 02-23-2018.csv...
Lendo: 03-01-2018.csv...
Lendo: 03-02-2018.csv...

Sucesso! Total de linhas carregadas: 3476825


## 3. Limpeza e Pré-processamento "Deep Learning Ready"
Redes neurais não aceitam valores infinitos ou texto. Vamos:
1. Converter colunas para numérico.
2. Tratar valores `Infinity` e `NaN`.
3. Transformar a coluna `Label` em números (0, 1, 2...).

In [16]:

# 1. Configuração de arquivos
folder = 'archive'
arquivos = ['02-14-2018.csv', '02-23-2018.csv', '03-01-2018.csv', '03-02-2018.csv']

print("--- INICIANDO PROCESSO ---")
df_list = []

# 2. Leitura Robusta (Lendo um por um e limpando nomes na hora)
for f in arquivos:
    path = os.path.join(folder, f)
    if os.path.exists(path):
        print(f"Lendo: {f}...")
        # Lemos o arquivo
        temp = pd.read_csv(path, low_memory=False)
        
        # Limpamos os espaços nos nomes das colunas IMEDIATAMENTE
        temp.columns = temp.columns.str.strip()
        
        # Removemos o Timestamp (não serve para a rede neural e gera erro de limpeza)
        if 'Timestamp' in temp.columns:
            temp.drop(columns=['Timestamp'], inplace=True)
            
        # Removemos linhas que repetem o cabeçalho (comum no dataset)
        temp = temp[temp.iloc[:, 0] != temp.columns[0]]
        
        df_list.append(temp)
    else:
        print(f"AVISO: Arquivo não encontrado: {path}")

# 3. Unificação
if len(df_list) > 0:
    df = pd.concat(df_list, ignore_index=True)
    del df_list # Libera memória dos arquivos individuais
    gc.collect()
    print(f"Total bruto carregado: {df.shape[0]} linhas.")
else:
    print("ERRO: Nenhum dado foi carregado. Verifique a pasta 'archive'.")

# 4. Limpeza de Dados Mortos (Infinitos e Nulos)
print("Tratando valores numéricos...")
cols_numericas = [c for c in df.columns if c != 'Label']
for col in cols_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# 5. Transformação da Label
le = LabelEncoder()
df['Label'] = df['Label'].astype(str).str.strip()
df['Label'] = le.fit_transform(df['Label'])

print("\n--- SUCESSO FINAL ---")
print(f"Classes detectadas: {le.classes_}")
print(f"Total de linhas prontas para o modelo: {df.shape[0]}")

gc.collect()

--- INICIANDO PROCESSO ---
Lendo: 02-14-2018.csv...
Lendo: 02-23-2018.csv...
Lendo: 03-01-2018.csv...
Lendo: 03-02-2018.csv...
Total bruto carregado: 3476825 linhas.
Tratando valores numéricos...

--- SUCESSO FINAL ---
Classes detectadas: ['Benign' 'Bot' 'Brute Force -Web' 'Brute Force -XSS' 'FTP-BruteForce'
 'Infilteration' 'SQL Injection' 'SSH-Bruteforce']
Total de linhas prontas para o modelo: 3460324


0

## 4. Divisão (Train, Validation, Test) e Escalonamento
Como temos um dataset massivo (~3,4 milhões de linhas), usaremos uma divisão de **60% para Treino**, **20% para Validação** (durante o treino) e **20% para Teste Final**.

Também aplicaremos o `StandardScaler` para que todas as colunas tenham a mesma escala, o que é fundamental para a rede neural convergir rapidamente.

In [ ]:


# 1. Separar Features (X) e Alvo (y)
X = df.drop('Label', axis=1)
y = df['Label']

# 2. Primeiro Split: 80% (Treino + Val) e 20% (Teste Final)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Segundo Split: Dos 80%, tiramos 25% para validação (25% de 80 = 20% do total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"Treino: {X_train.shape[0]} amostras")
print(f"Validação: {X_val.shape[0]} amostras")
print(f"Teste Final: {X_test.shape[0]} amostras")

# 4. Escalonamento (Normalização)
# Ajustamos o scaler APENAS nos dados de treino para evitar vazamento de dados (data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# 5. Limpeza de Memória (Crucial com 3.4M de linhas)
del df, X_train_val, y_train_val
gc.collect()

print("\n--- Dados prontos para a Rede Neural ---")

NameError: name 'df' is not defined